In [2]:
import json
from pathlib import Path
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    TableStructureOptions,
    TesseractCliOcrOptions,
)
from docling.document_converter import (
    DocumentConverter,
    PdfFormatOption,
)
from docling_core.types.doc import (
    ImageRefMode,
    PictureItem,
    TableItem,
)

## Configurations

In [3]:
PDF_PATH = Path(
    r"data\complex_rag_parsing_sample_with_sunny_image.pdf"
).resolve()

OUTPUT_DIR = Path(
    r"data\docling_parsed_output"
).resolve()

PAGE_IMAGE_DIR = OUTPUT_DIR / "page_images"
PICTURE_DIR = OUTPUT_DIR / "extracted_pictures"
TABLE_DIR = OUTPUT_DIR / "extracted_tables"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PAGE_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
PICTURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF not found: {PDF_PATH}")

In [4]:
pipeline_options = PdfPipelineOptions()

In [5]:
pipeline_options.do_ocr = True

In [6]:
pipeline_options.ocr_options = TesseractCliOcrOptions(
    tesseract_cmd=r"C:\Program Files\Tesseract-OCR\tesseract.exe",
    lang=["eng"],

    # False = selectable text + OCR where required
    # True = OCR complete pages
    force_full_page_ocr=False,
)

In [7]:
pipeline_options.do_table_structure = True

In [8]:
pipeline_options.table_structure_options = TableStructureOptions(
    do_cell_matching=True,
)

In [9]:
# Preserve page and picture images
pipeline_options.images_scale = 2.0
pipeline_options.generate_page_images = True
pipeline_options.generate_picture_images = True

## Document Converter

In [10]:
converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options
        )
    }
)

In [11]:
print("Parsing PDF with Docling...")

conversion_result = converter.convert(PDF_PATH)

Parsing PDF with Docling...


W1007 01:36:17.422000 9644 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


2026-10-07 01:36:36,682 MatchingPostProcessor WARNING  2 of 35 pdf cells matched neither a row nor a column band of the 3x3 grid and were dropped from the table


OSD failed (doc C:\Users\SAAD TARIQ\saadtariq-learning\generative_and_agentic_ai\Module_5_Retrieval_Augmented_Generation\data\complex_rag_parsing_sample_with_sunny_image.pdf, page: 0, OCR rectangle: 0, processed image file <tempfile._TemporaryFileWrapper object at 0x00000150555466F0>):
 b'Estimating resolution as 220\r\nWarning. Invalid resolution 0 dpi. Using 70 instead.\r\nToo few characters. Skipping this page\r\nError during processing.\r\n'


In [12]:
docling_document = conversion_result.document

print("PDF parsed successfully.")
print("Total pages:", len(docling_document.pages))
print("Total tables:", len(docling_document.tables))
print("Total pictures:", len(docling_document.pictures))

PDF parsed successfully.
Total pages: 24
Total tables: 23
Total pictures: 10


## Writing Text

In [13]:
json_output_path = OUTPUT_DIR / "docling_document.json"

In [14]:
json_output_path.write_text(
    json.dumps(
        docling_document.export_to_dict(),
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("JSON saved:", json_output_path)

JSON saved: C:\Users\SAAD TARIQ\saadtariq-learning\generative_and_agentic_ai\Module_5_Retrieval_Augmented_Generation\data\docling_parsed_output\docling_document.json


In [15]:
markdown_output_path = OUTPUT_DIR / "rag_ready_document.md"

docling_document.save_as_markdown(
    markdown_output_path,
    image_mode=ImageRefMode.REFERENCED,
)

print("Markdown saved:", markdown_output_path)

Markdown saved: C:\Users\SAAD TARIQ\saadtariq-learning\generative_and_agentic_ai\Module_5_Retrieval_Augmented_Generation\data\docling_parsed_output\rag_ready_document.md


In [16]:
text_output_path = OUTPUT_DIR / "extracted_text.txt"

text_output_path.write_text(
    docling_document.export_to_markdown(strict_text=True),
    encoding="utf-8",
)

print("Plain text saved:", text_output_path)

Parameter `strict_text` has been deprecated and will be ignored.


Plain text saved: C:\Users\SAAD TARIQ\saadtariq-learning\generative_and_agentic_ai\Module_5_Retrieval_Augmented_Generation\data\docling_parsed_output\extracted_text.txt


## Saving Results

In [17]:
for page_number, page in docling_document.pages.items():

    if page.image is None:
        continue

    page_image_path = (
        PAGE_IMAGE_DIR
        / f"page_{page.page_no:03d}.png"
    )

    page.image.pil_image.save(
        page_image_path,
        format="PNG",
    )

print("Page images saved:", PAGE_IMAGE_DIR)

Page images saved: C:\Users\SAAD TARIQ\saadtariq-learning\generative_and_agentic_ai\Module_5_Retrieval_Augmented_Generation\data\docling_parsed_output\page_images


In [18]:
for table_index, table in enumerate(
    docling_document.tables,
    start=1,
):
    table_df = table.export_to_dataframe(
        doc=docling_document
    )

    csv_path = TABLE_DIR / f"table_{table_index:03d}.csv"
    markdown_path = TABLE_DIR / f"table_{table_index:03d}.md"
    image_path = TABLE_DIR / f"table_{table_index:03d}.png"

    table_df.to_csv(
        csv_path,
        index=False,
        encoding="utf-8",
    )

    markdown_path.write_text(
        table_df.to_markdown(index=False),
        encoding="utf-8",
    )

    table_image = table.get_image(docling_document)

    if table_image is not None:
        table_image.save(image_path, format="PNG")

print("Tables saved:", TABLE_DIR)


Tables saved: C:\Users\SAAD TARIQ\saadtariq-learning\generative_and_agentic_ai\Module_5_Retrieval_Augmented_Generation\data\docling_parsed_output\extracted_tables


In [19]:
picture_counter = 0

for element, _level in docling_document.iterate_items():

    if not isinstance(element, PictureItem):
        continue

    picture_counter += 1

    picture_path = (
        PICTURE_DIR
        / f"picture_{picture_counter:03d}.png"
    )

    picture_image = element.get_image(docling_document)

    if picture_image is not None:
        picture_image.save(
            picture_path,
            format="PNG",
        )

print("Pictures saved:", PICTURE_DIR)

Pictures saved: C:\Users\SAAD TARIQ\saadtariq-learning\generative_and_agentic_ai\Module_5_Retrieval_Augmented_Generation\data\docling_parsed_output\extracted_pictures
